<a href="https://colab.research.google.com/github/gfky1356-ship-it/SGX-data-download/blob/main/Final_SP500_Q_report_scan_by_Gemini_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cell 1: 挂载网盘与初始化环境

In [ ]:
from google.colab import drive
import os
from datetime import datetime
import pytz # 引入时区库

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 生成当前时间戳 (强制转换为新加坡时间，格式为：年月日_时分)
# 注意：文件名中不能使用冒号(:)，所以使用 1137 这样的格式
sgt_tz = pytz.timezone('Asia/Singapore')
timestamp = datetime.now(sgt_tz).strftime("%Y%m%d_%H%M")

# 3. 创建专用主文件夹
base_dir = '/content/drive/MyDrive/SP500_Financial_Analysis'
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    print(f"已创建主文件夹: {base_dir}")
else:
    print(f"主文件夹已就绪: {base_dir}")

print(f"✅ 当前设定的时间戳标签为: {timestamp}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
主文件夹已就绪: /content/drive/MyDrive/SP500_Financial_Analysis
✅ 当前设定的时间戳标签为: 20260527_1139


### Cell 2: 第一层与第二层扫描 (已放宽参数，并将参数提取到顶部方便修改)

In [ ]:
!pip install pandas yfinance lxml requests

import pandas as pd
import yfinance as yf
import numpy as np
import requests
from io import StringIO

# ==========================================
# 🎯 您可以在这里轻松修改筛选条件：
MIN_PE = 0             # 最小市盈率 (大于0)
MIN_ROE = 0.10         # 最小净资产收益率 (0.10 = 10%)
MIN_GM_GROWTH = 0.00   # 最小毛利率平均环比增长 (0.00 表示只要毛利率整体没有倒退即可)
# ==========================================

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
html_data = StringIO(response.text)
sp500_df = pd.read_html(html_data)[0]
tickers = sp500_df['Symbol'].str.replace('.', '-').tolist()

filtered_stocks = []
analysis_data = []

print(f"🚀 开始第一层 & 第二层扫描：")
print(f"当前条件: PE > {MIN_PE}, ROE > {MIN_ROE*100}%, 毛利平均增长 >= {MIN_GM_GROWTH*100}%\n")

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        pe = info.get('trailingPE', 0)
        roe = info.get('returnOnEquity', 0)
        if pe is None or pe <= MIN_PE or roe is None or roe <= MIN_ROE:
            continue

        financials = stock.quarterly_financials
        if financials is not None and not financials.empty:
            if 'Gross Profit' in financials.index and 'Total Revenue' in financials.index:
                gross_profit = financials.loc['Gross Profit'][:4]
                total_revenue = financials.loc['Total Revenue'][:4]

                if len(gross_profit) == 4 and not total_revenue.isna().any():
                    gm = (gross_profit / total_revenue).values
                    gm_growths = [(gm[i] - gm[i+1]) / gm[i+1] for i in range(3) if gm[i+1] != 0]

                    if gm_growths:
                        avg_gm_growth = np.mean(gm_growths)

                        if avg_gm_growth >= MIN_GM_GROWTH:
                            de_ratio = info.get('debtToEquity', 0)
                            filtered_stocks.append(ticker)
                            analysis_data.append({
                                'Company': info.get('shortName', ticker),
                                'Ticker': ticker,
                                'Q4_GM (Latest)': round(gm[0], 4),
                                'Q3_GM': round(gm[1], 4),
                                'Q2_GM': round(gm[2], 4),
                                'Q1_GM (Oldest)': round(gm[3], 4),
                                'Avg_GM_Growth': round(avg_gm_growth, 4),
                                'TTM_ROE': round(roe, 4),
                                'PE': round(pe, 2),
                                'DE_Ratio': round(de_ratio, 2) if de_ratio else 'N/A'
                            })
                            print(f"⭐ 命中目标: {ticker}")
    except Exception as e:
        pass # 忽略解析异常

df_results = pd.DataFrame(analysis_data)
print(f"\n🎉 扫描完成！共筛选出 {len(filtered_stocks)} 只符合条件的股票。")


🚀 开始第一层 & 第二层扫描：
当前条件: PE > 0, ROE > 10.0%, 毛利平均增长 >= 0.0%

⭐ 命中目标: ADBE
⭐ 命中目标: A
⭐ 命中目标: ALGN
⭐ 命中目标: GOOGL
⭐ 命中目标: GOOG
⭐ 命中目标: AMZN
⭐ 命中目标: AEE
⭐ 命中目标: AMT
⭐ 命中目标: AME
⭐ 命中目标: AMGN
⭐ 命中目标: APH
⭐ 命中目标: AON
⭐ 命中目标: APA
⭐ 命中目标: AAPL
⭐ 命中目标: AMAT
⭐ 命中目标: APP
⭐ 命中目标: ADSK
⭐ 命中目标: ADP
⭐ 命中目标: AVY
⭐ 命中目标: BLK
⭐ 命中目标: BSX
⭐ 命中目标: AVGO
⭐ 命中目标: BRO
⭐ 命中目标: BF-B
⭐ 命中目标: CASY
⭐ 命中目标: CAT
⭐ 命中目标: CBOE
⭐ 命中目标: CDW
⭐ 命中目标: COR
⭐ 命中目标: CHTR
⭐ 命中目标: CHD
⭐ 命中目标: CTAS
⭐ 命中目标: CSCO
⭐ 命中目标: CME
⭐ 命中目标: CMS
⭐ 命中目标: KO
⭐ 命中目标: CL
⭐ 命中目标: FIX
⭐ 命中目标: COP
⭐ 命中目标: CEG
⭐ 命中目标: GLW
⭐ 命中目标: CPAY
⭐ 命中目标: CSX
⭐ 命中目标: CMI
⭐ 命中目标: DECK
⭐ 命中目标: DXCM
⭐ 命中目标: DLTR
⭐ 命中目标: EBAY
⭐ 命中目标: EIX
⭐ 命中目标: EW
⭐ 命中目标: EA
⭐ 命中目标: EMR
⭐ 命中目标: ETR
⭐ 命中目标: EOG
⭐ 命中目标: EQT
⭐ 命中目标: ES
⭐ 命中目标: EXE
⭐ 命中目标: EXPD
⭐ 命中目标: FFIV
⭐ 命中目标: FSLR
⭐ 命中目标: GRMN
⭐ 命中目标: IT
⭐ 命中目标: GD
⭐ 命中目标: GILD
⭐ 命中目标: GDDY
⭐ 命中目标: HSY
⭐ 命中目标: HON
⭐ 命中目标: HST
⭐ 命中目标: HWM
⭐ 命中目标: HII
⭐ 命中目标: IDXX
⭐ 命中目标: IBKR
⭐ 命中目标: ICE
⭐ 命中目标: INTU
⭐ 命中目标: JBL
⭐ 命中目标: KVUE
⭐ 命中目

### Cell 3: 第三层扫描 (下载精选名单的 SEC 财报)

In [ ]:
!pip install sec-edgar-downloader

from sec_edgar_downloader import Downloader

# ⚠️ 请在此处替换为您真实的邮箱地址
dl = Downloader("MyDataAnalysis", "your.email@example.com", base_dir)

print(f"📥 开始第三层：仅为筛选出的 {len(filtered_stocks)} 家公司下载/更新 SEC 财报...")

if len(filtered_stocks) > 0:
    for ticker in filtered_stocks:
        try:
            print(f"检查并获取 [{ticker}] 的最新财报...")
            dl.get("10-K", ticker, limit=1)
            dl.get("10-Q", ticker, limit=3)
        except Exception as e:
            print(f"[{ticker}] SEC 获取失败: {e}")
    print("\n✅ SEC 财报更新完毕。")
else:
    print("没有符合条件的股票，跳过下载步骤。")


📥 开始第三层：仅为筛选出的 137 家公司下载/更新 SEC 财报...
检查并获取 [ADBE] 的最新财报...
检查并获取 [A] 的最新财报...
检查并获取 [ALGN] 的最新财报...
检查并获取 [GOOGL] 的最新财报...
检查并获取 [GOOG] 的最新财报...
检查并获取 [AMZN] 的最新财报...
检查并获取 [AEE] 的最新财报...
检查并获取 [AMT] 的最新财报...
检查并获取 [AME] 的最新财报...
检查并获取 [AMGN] 的最新财报...
检查并获取 [APH] 的最新财报...
检查并获取 [AON] 的最新财报...
检查并获取 [APA] 的最新财报...
检查并获取 [AAPL] 的最新财报...
检查并获取 [AMAT] 的最新财报...
检查并获取 [APP] 的最新财报...
检查并获取 [ADSK] 的最新财报...
检查并获取 [ADP] 的最新财报...
检查并获取 [AVY] 的最新财报...
检查并获取 [BLK] 的最新财报...
检查并获取 [BSX] 的最新财报...
检查并获取 [AVGO] 的最新财报...
检查并获取 [BRO] 的最新财报...
检查并获取 [BF-B] 的最新财报...
检查并获取 [CASY] 的最新财报...
检查并获取 [CAT] 的最新财报...
检查并获取 [CBOE] 的最新财报...
检查并获取 [CDW] 的最新财报...
检查并获取 [COR] 的最新财报...
检查并获取 [CHTR] 的最新财报...
检查并获取 [CHD] 的最新财报...
检查并获取 [CTAS] 的最新财报...
检查并获取 [CSCO] 的最新财报...
检查并获取 [CME] 的最新财报...
检查并获取 [CMS] 的最新财报...
检查并获取 [KO] 的最新财报...
检查并获取 [CL] 的最新财报...
检查并获取 [FIX] 的最新财报...
检查并获取 [COP] 的最新财报...
检查并获取 [CEG] 的最新财报...
检查并获取 [GLW] 的最新财报...
检查并获取 [CPAY] 的最新财报...
检查并获取 [CSX] 的最新财报...
检查并获取 [CMI] 的最新财报...
检查并获取 [DECK] 的最新财报...
检查

### Cell 4: 导出报告与观察列表

In [ ]:
!pip install xlsxwriter yfinance
import os
import yfinance as yf
import pandas as pd
import numpy as np
import time

excel_filepath = os.path.join(base_dir, f"Stock_Analysis_Top20_Growth_{timestamp}.xlsx")
tv_filepath_all = os.path.join(base_dir, f"TradingView_Top20_{timestamp}.txt")
tv_filepath_uptrend = os.path.join(base_dir, f"TradingView_SteadyUptrend_{timestamp}.txt")

if not df_results.empty:
    df_export = df_results.copy()

    # --- 1. 数据处理：计算平均值、排序 ---
    df_export['Avg_4Q_GM'] = df_export[['Q4_GM (Latest)', 'Q3_GM', 'Q2_GM', 'Q1_GM (Oldest)']].mean(axis=1)
    df_export = df_export.sort_values(by='Avg_GM_Growth', ascending=False).reset_index(drop=True)
    top_n = min(20, len(df_export))

    # --- 2. 获取年度毛利率 (Yearly GM) ---
    print(f"正在为毛利增长排名前 {top_n} 强的股票抓取上一年度的毛利率数据，请稍候...")
    yearly_gms = []
    for i in range(top_n):
        ticker = df_export.iloc[i]['Ticker']
        try:
            stock = yf.Ticker(ticker)
            annuals = stock.financials
            if not annuals.empty and 'Gross Profit' in annuals.index and 'Total Revenue' in annuals.index:
                gp = annuals.loc['Gross Profit'].iloc[0]
                tr = annuals.loc['Total Revenue'].iloc[0]
                yearly_gm = gp / tr
            else:
                yearly_gm = np.nan
        except:
            yearly_gm = np.nan
        yearly_gms.append(yearly_gm)

    all_yearly_gms = yearly_gms + [np.nan] * (len(df_export) - top_n)
    df_export['Yearly_GM (Last FY)'] = all_yearly_gms

    cols = [
        'Company', 'Ticker',
        'Yearly_GM (Last FY)', 'Q1_GM (Oldest)', 'Q2_GM', 'Q3_GM', 'Q4_GM (Latest)',
        'Avg_4Q_GM', 'Avg_GM_Growth', 'TTM_ROE', 'PE', 'DE_Ratio'
    ]
    df_export = df_export[cols]

    # --- 3. 写入 Excel ---
    writer = pd.ExcelWriter(excel_filepath, engine='xlsxwriter')
    df_export.to_excel(writer, sheet_name='Filtered_Stocks', index=False)
    workbook = writer.book
    worksheet = writer.sheets['Filtered_Stocks']

    # --- 4. 样式与格式化 ---
    percent_fmt = workbook.add_format({'num_format': '0.00%'})
    pink_fmt = workbook.add_format({'bg_color': '#FFC0CB'})
    pink_percent_fmt = workbook.add_format({'num_format': '0.00%', 'bg_color': '#FFC0CB'})

    worksheet.set_column('A:A', 25)
    worksheet.set_column('B:B', 10)
    worksheet.set_column('C:J', 15, percent_fmt)
    worksheet.set_column('K:L', 12)

    # 为 Top 20 行应用粉色高亮
    for row in range(1, top_n + 1):
        worksheet.write(row, 0, df_export.iloc[row-1, 0], pink_fmt)
        worksheet.write(row, 1, df_export.iloc[row-1, 1], pink_fmt)
        for col in range(2, 10):
            val = df_export.iloc[row-1, col]
            if pd.isna(val):
                worksheet.write(row, col, "N/A", pink_fmt)
            else:
                worksheet.write(row, col, val, pink_percent_fmt)
        for col in range(10, 12):
            worksheet.write(row, col, df_export.iloc[row-1, col], pink_fmt)

    # --- 5. 生成 Top 20 独立混合柱状图 (🌟 含明黄色高亮过滤) ---
    chart_row = 2
    for i in range(top_n):
        ticker = df_export.iloc[i]['Ticker']

        q1 = df_export.iloc[i]['Q1_GM (Oldest)']
        q2 = df_export.iloc[i]['Q2_GM']
        q3 = df_export.iloc[i]['Q3_GM']
        q4 = df_export.iloc[i]['Q4_GM (Latest)']

        # 判断是否属于“稳步上升”的神仙形态
        is_steady_uptrend = (q1 < q2) and (q2 < q3) and (q3 < q4)

        chart = workbook.add_chart({'type': 'column'})

        chart.add_series({
            'name':       ['Filtered_Stocks', i+1, 0],
            'categories': ['Filtered_Stocks', 0, 2, 0, 6],
            'values':     ['Filtered_Stocks', i+1, 2, i+1, 6],
            'points': [
                {'fill': {'color': 'red'}},
                {'fill': {'color': '#4F81BD'}},
                {'fill': {'color': '#4F81BD'}},
                {'fill': {'color': '#4F81BD'}},
                {'fill': {'color': '#4F81BD'}},
            ],
            'data_labels': {'value': True, 'num_format': '0.0%'}
        })

        # 🎯 修改高亮颜色：使用对比度更高的明黄色
        if is_steady_uptrend:
            chart.set_title ({'name': f'🌟 {ticker} - 强劲连涨 (Steady Uptrend)'})
            chart.set_chartarea({'fill': {'color': '#FFFF00'}}) # 明黄色
        else:
            chart.set_title ({'name': f'{ticker} - Yearly & 4Q Gross Margin'})

        chart.set_legend({'none': True})
        chart.set_y_axis({'num_format': '0%'})
        chart.set_size({'width': 480, 'height': 280})

        if i % 2 == 0:
            pos = f'N{chart_row}'
        else:
            pos = f'W{chart_row}'
            chart_row += 15

        worksheet.insert_chart(pos, chart)
    writer.close()
    print(f"✅ 高级分析 Excel (含连续上涨高对比度黄底高亮图表) 已保存至:\n{excel_filepath}\n")

    # --- 6. 导出两个 TradingView 观察列表 ---
    top_tickers = df_export['Ticker'].head(top_n).tolist()
    with open(tv_filepath_all, 'w') as f:
        f.write(",".join(top_tickers))
    print(f"✅ TV Watchlist 1 (完整 Top {top_n}) 已保存至:\n{tv_filepath_all}")

    uptrend_df = df_export.head(top_n)[
        (df_export['Q1_GM (Oldest)'] < df_export['Q2_GM']) &
        (df_export['Q2_GM'] < df_export['Q3_GM']) &
        (df_export['Q3_GM'] < df_export['Q4_GM (Latest)'])
    ]
    uptrend_tickers = uptrend_df['Ticker'].tolist()

    if uptrend_tickers:
        with open(tv_filepath_uptrend, 'w') as f:
            f.write(",".join(uptrend_tickers))
        print(f"✅ TV Watchlist 2 (严苛筛选：连续上升 🌟 共 {len(uptrend_tickers)} 只) 已保存至:\n{tv_filepath_uptrend}\n")
    else:
        print("\n⚠️ 在 Top 20 中，没有发现连续四个季度毛利率【每一季都严格高于上一季】的公司。因此未生成名单2。\n")

    # --- 7. 🧹 自动清理 7 天前的历史报告文件 ---
    print("开始清理过期文件 (7天前)...")
    current_time = time.time()
    seven_days_in_seconds = 7 * 24 * 60 * 60
    deleted_count = 0

    for filename in os.listdir(base_dir):
        # 仅清理我们生成的特定格式文件，避免误删 SEC 财报文件夹
        if filename.startswith("Stock_Analysis_") or filename.startswith("TradingView_"):
            file_path = os.path.join(base_dir, filename)
            if os.path.isfile(file_path):
                # 获取文件的最后修改时间
                file_mtime = os.path.getmtime(file_path)
                # 如果文件超过7天
                if (current_time - file_mtime) > seven_days_in_seconds:
                    os.remove(file_path)
                    print(f"🗑️ 已删除过期文件: {filename}")
                    deleted_count += 1

    if deleted_count == 0:
        print("✅ 检查完毕，没有需要清理的过期文件。")
    else:
        print(f"✅ 清理完成，共删除 {deleted_count} 个过期文件。")

else:
    print("未生成文件，因为名单为空。请回到 Cell 2 放宽筛选条件再试。")

正在为毛利增长排名前 20 强的股票抓取上一年度的毛利率数据，请稍候...
✅ 高级分析 Excel (含连续上涨高对比度黄底高亮图表) 已保存至:
/content/drive/MyDrive/SP500_Financial_Analysis/Stock_Analysis_Top20_Growth_20260527_1139.xlsx

✅ TV Watchlist 1 (完整 Top 20) 已保存至:
/content/drive/MyDrive/SP500_Financial_Analysis/TradingView_Top20_20260527_1139.txt
✅ TV Watchlist 2 (严苛筛选：连续上升 🌟 共 11 只) 已保存至:
/content/drive/MyDrive/SP500_Financial_Analysis/TradingView_SteadyUptrend_20260527_1139.txt

开始清理过期文件 (7天前)...
✅ 检查完毕，没有需要清理的过期文件。


/tmp/ipykernel_919/3785022599.py:133: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  uptrend_df = df_export.head(top_n)[
